# Laboratorio 4 — Enunciado: Temperaturas CRU

## Antes de empezar

El proyecto ya está preparado. Trabajen en parejas y ejecuten los comandos
desde la carpeta de este laboratorio:

```bash
uv sync
uv run jupyter lab
```

El notebook es el espacio para explorar los datos y probar las
transformaciones. El código definitivo debe quedar en `src/meteolab/`.

## Flujo de trabajo

En cada operación seguirán este ciclo:

1. Investigar la operación o el método que necesitan.
2. Probarlo directamente sobre los datos en el notebook.
3. Observar y comprobar el resultado.
4. Trasladar la solución al módulo indicado.
5. Comprobarla con `uv run pytest -m etapaN`.
6. Interpretar los resultados y responder la pregunta de la etapa.

## El dataset CRU

El archivo `data/cru_country_tmp_tidy.csv` contiene temperaturas medias del
conjunto de datos de la *Climatic Research Unit* (CRU). Cada fila representa
un país, un año y un período. El período puede ser un mes (`JAN` a `DEC`),
una estación climática (`DJF`, `MAM`, `JJA`, `SON`) o el promedio anual
(`ANN`).

| Columna | Significado | Tipo esperado |
|---|---|---|
| `country` | nombre del país | `String` |
| `iso_alpha2` | código ISO de dos letras | `String` |
| `iso_alpha3` | código ISO de tres letras | `String` |
| `year` | año de la observación | `Int64` |
| `period` | mes, estación climática o promedio anual | `String` |
| `temperature_c` | temperatura media en grados Celsius | `Float64` |
| `parameter` | indicador medido | `String` |
| `units` | unidad del indicador | `String` |
| `source_file` | archivo de origen | `String` |

El laboratorio se concentrará en las **temperaturas medias mensuales**.
Durante la limpieza descartarán las filas de `DJF`, `MAM`, `JJA`, `SON` y
`ANN`. Desde ese punto, ninguna agregación podrá usar esas observaciones.

```mermaid
flowchart LR
    A["CSV CRU<br/>17 períodos"] --> B["Exploración"]
    B --> C["Lectura y esquema"]
    C --> D["Limpieza<br/>solo JAN–DEC"]
    D --> E["Fechas mensuales"]
    E --> F["Agregaciones y ventanas"]
    F --> G["Pipeline lazy"]
```

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — Polars</strong>
  <strong>Polars</strong> es una librería de Python para trabajar con datos tabulares. Sus estructuras principales son <code>DataFrame</code>, que contiene datos ya materializados, y <code>LazyFrame</code>, que representa una consulta aún no ejecutada. Polars ofrece una API de expresiones para describir transformaciones sobre columnas y permite trabajar en modo <em>eager</em> o <em>lazy</em>.
</div>

## Evaluación

| Etapa | Contenido | Implementación | Análisis y preguntas | Total |
|---|---|---:|---:|---:|
| 1 | Exploración del CSV | 0.2 | 0.4 | 0.6 |
| 2 | Polars e I/O | 0.4 | 0.3 | 0.7 |
| 3 | Esquema y validación | 0.5 | 0.3 | 0.8 |
| 4 | Limpieza y selección mensual | 0.5 | 0.4 | 0.9 |
| 5 | Fechas y agregaciones | 0.7 | 0.4 | 1.1 |
| 6 | Ventanas y anomalías | 0.5 | 0.3 | 0.8 |
| 7 | Pipeline lazy y análisis | 0.3 | 0.8 | 1.1 |
| **Total** | | **3.1** | **2.9** | **6.0** |

## Preparación

In [ ]:
import sys
from pathlib import Path

import plotly.express as px
import polars as pl

RAIZ = Path.cwd() if Path("pyproject.toml").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

RUTA_DATOS = RAIZ / "data"
RUTA_CSV = RUTA_DATOS / "cru_country_tmp_tidy.csv"

print("Proyecto:", RAIZ)
print("Archivo :", RUTA_CSV)

---
# Etapa 1 — Explorar el archivo (0.6 puntos)

Antes de decidir cómo leer o transformar una tabla, observen sus dimensiones,
sus tipos y sus valores ausentes. Esta etapa no modifica los datos: levanta
evidencia para las decisiones que tomarán después.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — tabla y esquema</strong>
  Una tabla organiza observaciones en filas y variables en columnas. El <strong>esquema</strong> es la lista de columnas junto con sus tipos. En este archivo, el esquema distingue identificadores de texto, años enteros y temperaturas decimales.
</div>

### 1.1 — Leer e inspeccionar (0.1 puntos)

Lean el CSV sin imponer todavía el esquema. En la siguiente etapa justificarán
qué tipos y valores nulos deben declarar explícitamente.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — CSV</strong>
  <strong>CSV</strong> (*Comma-Separated Values*) es un archivo de texto en el que cada fila representa un registro y las columnas se separan mediante un delimitador, normalmente una coma. Es fácil de intercambiar, pero no guarda de forma completa el esquema de la tabla: al leerlo, la librería debe inferir o recibir los tipos de cada columna.
</div>

Investiguen `polars.read_csv` y utilícenla para leer `RUTA_CSV`.

In [ ]:
raw = pl.read_csv(RUTA_CSV)

Inspeccionen las primeras y las últimas diez filas. Luego, muestren diez filas
aleatorias con una semilla fija.

In [ ]:
display(raw.head(10))
display(raw.tail(10))
display(raw.sample(n=10, seed=7202))

Revisen las dimensiones, los nombres de las columnas, el esquema y una vista
compacta de los valores.

In [ ]:
print("Dimensiones:", raw.shape)
print("Columnas:", raw.columns)
print("Esquema:")
print(raw.schema)
raw.glimpse()

Generen un resumen estadístico y cuenten los valores nulos por columna.

In [ ]:
display(raw.describe())
display(raw.null_count())

### 1.2 — Visualizar la distribución (0.1 puntos)

Plotly Express permite construir gráficos interactivos a partir de columnas
tabulares. Exploren la distribución de `temperature_c` sin confundir los
valores ausentes con una temperatura.

In [ ]:
fig = px.histogram(
    raw,
    x="temperature_c",
    nbins=40,
    title="Distribución de las temperaturas CRU",
    labels={"temperature_c": "Temperatura (°C)"},
)
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 1 — Qué muestra el archivo (0.2 puntos)</strong>
  ¿Cuántas filas y columnas tiene el CSV? ¿Qué columnas son identificadores, cuáles representan tiempo y cuál contiene la medición? Expliquen qué información se pierde si `temperature_c` se lee como texto.
</div>

<code>Escribe tu respuesta aquí:</code> El dataset contiene 408000 filas y 9 columnas. Respecto a los identificadores pensandolos como "llave" para reconocer la medición, me imagino que para determinar la medición se necesita el país (country) o en su defecto las abreviaciones tipo iso_alpha2 o iso_alpha3, luego se necesitaría la fecha de medición que se obtiene con year y period y finalmente la descripción parameter y utils para saber cómo se hizo la medición y caracterizarla. Source_file podría ser identificador por si solo, si hubiese suficiente información de la columna, sin embargo no dice algo que la describa de forma tangible como las otras. Year y period en principio muestran tiempo (al menos los meses en period). La medición misma está en temperature_c. Si escribieramos esta última como texto, en principio perderíamos habilidades como la recién realizada de hacer histogramas considerando rango, pero también otras como suma, resta, producto, mínimos, máximos entre, muchos otros.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 2 — Escalas temporales (0.2 puntos)</strong>
  El archivo contiene 17 valores distintos en `period`. ¿Por qué no se deben mezclar en un mismo promedio las filas mensuales, estacionales y anuales? Anticipen qué período conservarán durante la limpieza y por qué.
</div>

<code>Escribe tu respuesta aquí:</code> Me arriesgo a pensar que tienen nociones temporales distintas, aunque sean en un mismo año no son comparables entre si y si quisiera hacer promedios o crecimientos, incluir el promedio anual y estacional pone ruido no cuantificable. Estimo que el periodo a conservar es ANN que es el promedio anual, lo relevante para el objetivo. 

---
# Etapa 2 — Polars e I/O (0.7 puntos)

En esta etapa pasarán de una lectura exploratoria a una lectura reproducible.
Usarán el CSV como fuente de entrada y compararán la lectura eager con la
lectura lazy.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — eager y lazy</strong>
  En modo <em>eager</em>, una operación se ejecuta cuando se llama y devuelve un <code>DataFrame</code>. En modo <em>lazy</em>, las operaciones construyen un plan y se ejecutan al llamar <code>collect()</code>. <code>scan_csv</code> permite describir una consulta sin cargar de inmediato todo el archivo.
</div>

### 2.1 — Investigar y probar la lectura CSV (0.3 puntos)

Investiguen `schema_overrides` en `read_csv`. Luego, vuelvan a leer el CSV
declarando `year` como `Int64` y `temperature_c` como `Float64`. Comparen el
esquema de esta tabla con el de `raw`.

In [ ]:
lecturas = pl.read_csv(RUTA_CSV, schema_overrides={
    "year": pl.Int64,
    "temperature_c": pl.Float64
})

In [ ]:
print("Esquema inferido :", raw.schema)
print("Esquema declarado:", lecturas.schema)

Investiguen `scan_csv` y construyan una consulta que filtre algunos países
sin ejecutarla. Comprueben el tipo de objeto y materialicen el resultado con
`collect()`.

In [ ]:
consulta = pl.scan_csv(RUTA_CSV).filter((pl.col("country")=="China")|(pl.col("country")=="Chile"))

In [ ]:
print(type(consulta))
display(consulta.collect().head())
print(consulta.explain())

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 3 — Elegir la lectura (0.3 puntos)</strong>
  ¿Qué ventaja ofrece declarar `year` y `temperature_c` al leer el CSV? ¿En qué situación tendría sentido usar `scan_csv` en vez de `read_csv`?
</div>

<code>Escribe tu respuesta aquí:</code> En cuanto a las ventajas, si fuesen leídas como string sería bastante complejo hacer comparaciones porque los string no tienen el buen orden que los numeros poseen (o no de una forma tan directa), entonces hacer comparaciones como year>=1900 serían extrañas. Poder declarar permite aprovechar la estructura. Por otro lado, scan_csv parece no cargar toda la información de inmediato, en el caso anterior si sólo quisieramos extraer esos dos países se hace todo más rápido. 

### 2.2 — Trasladar la lectura CSV al módulo (0.1 puntos)

Después de probar las operaciones, implementen en `src/meteolab/carga.py`:

- `leer_temperaturas(ruta)`, con `schema_overrides`;
- `escanear_temperaturas(ruta)`, que debe devolver un `LazyFrame`;
Comprueben también que `escanear_temperaturas` devuelve un `LazyFrame` y que
la consulta se materializa solo con `collect()`.

In [ ]:
from src.meteolab.carga import (
    escanear_temperaturas,
    leer_temperaturas,
)

lecturas_modulo = leer_temperaturas(RUTA_CSV)
consulta_modulo = escanear_temperaturas(RUTA_CSV)

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240, 136, 62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Al módulo — <code>src/meteolab/carga.py</code></strong>
  Trasladen las funciones de lectura al módulo. Comprueben con <code>uv run pytest -m etapa2</code> que los tipos, los nulos y el modo lazy se comporten como en el notebook.
</div>

---
# Etapa 3 — Tipos y validación (0.8 puntos)

Leer un archivo no basta: hay que comprobar que los nombres, tipos y valores
permitidos cumplen el contrato del dataset.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — inferir, forzar y validar</strong>
  <strong>Inferir</strong> es dejar que la librería decida un tipo a partir de los valores observados. <strong>Forzar</strong> es declarar el tipo esperado al leer o transformar. <strong>Validar</strong> es comprobar que, además del tipo, los valores cumplen restricciones del dominio.
</div>

### 3.1 — Investigar y declarar el esquema (0.3 puntos)

Investiguen los tipos de Polars y construyan en el notebook un diccionario
con el esquema esperado. Comparen ese diccionario con `lecturas.schema` y
expliquen cualquier diferencia.

In [ ]:
esquema_notebook = {
    "country": pl.String,
    "iso_alpha2": pl.String,
    "iso_alpha3": pl.String,
    "year": pl.Int64,
    "period": pl.String,
    "temperature_c": pl.Float64,
    "parameter": pl.String,
    "units": pl.String,
    "source_file": pl.String
}

In [ ]:
diferencias = {
    columna: (lecturas.schema.get(columna), tipo)
    for columna, tipo in esquema_notebook.items()
    if lecturas.schema.get(columna) != tipo
}
print("Diferencias:", diferencias)

Investiguen Pandera y definan las restricciones que corresponden a este
dataset: años entre 1901 y 2025, períodos permitidos, `Mean Temperature` y
`degrees Celsius`.

In [ ]:
from src.meteolab.constantes import PERIODOS_VALIDOS

In [ ]:
import pandera.polars as pa

In [ ]:
validacion_notebook = pa.DataFrameSchema(
    {
        "country": pa.Column(pl.String),
        "iso_alpha2": pa.Column(pl.String),
        "iso_alpha3": pa.Column(pl.String),
        "year": pa.Column(
            pl.Int64, pa.Check.in_range(min_value=1901, max_value=2025)
        ),
        "period": pa.Column(pl.String, pa.Check.isin(PERIODOS_VALIDOS)),
        "temperature_c": pa.Column(pl.Float64,nullable=True),
        "parameter": pa.Column(
            pl.String, pa.Check.equal_to("Mean Temperature")
        ),
        "units": pa.Column(pl.String, pa.Check.equal_to("degrees Celsius")),
        "source_file": pa.Column(pl.String),
    }
)

Los 17 valores de `period` pertenecen al contrato del CSV. Que existan en el
esquema no significa que todos vayan a entrar al análisis: la selección de
períodos mensuales se hará en la limpieza.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 4 — El contrato y el análisis (0.2 puntos)</strong>
  ¿Por qué conviene aceptar `DJF`, `MAM`, `JJA` y `ANN` al validar el archivo, pero excluirlos después durante la limpieza? Relacionen la respuesta con la diferencia entre validar una fuente y definir el universo del análisis.
</div>

<code>Escribe tu respuesta aquí:</code> En primer lugar, es posible validar la lógica general del archivo, es decir que vengan los archivos en un tipo de formato o schema y trabajar de buena manera con los types. Esto es relevante porque si después necesitara analizar algún elemento con esas opciones que luego se van a eliminar, puedo tener el sosten de hacerlo con tranquilidad sin que los tipos cambien o asegurando que se hizo una buena lectura general del archivo; eso lo hace una fuente validada. El universo de análisis es efectivamente más pequeño y necesito que las columnas estén correctamente formateadas para usarlas, pero eso es un efecto secundario de lo anterior y siempre tendré la opción de incluir más en la medida que la fuente lo permita.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 5 — Validar tipos y valores (0.1 puntos)</strong>
  ¿Qué aporta Pandera, además de comparar `lecturas.schema` con el esquema esperado? Mencionen una restricción de valores que Pandera pueda comprobar en este dataset.
</div>

<code>Escribe tu respuesta aquí:</code> Pandera permite hacer muchos más checks al esquema, un ejemplo podría ser revisar en iso_alpha2 que sea de largo 2, iso_alpha largo 3 (usando <code>pa.Check.str_length(min=2,max=2)</code> en el primer caso y 3 en vez de dos en el segundo caso). Otra opción sería revisar que fuesen solo letras mayusculas usando <code>pa.Check.str_matches(r"^[A-Z]+$")</code>

### 3.2 — Trasladar la validación al módulo (0.2 puntos)

Después de probar el esquema y sus restricciones, implementen en
`src/meteolab/esquema.py`:

- `comparar_esquema`, para informar columnas faltantes y tipos distintos;
- `validar_esquema`, para rechazar un esquema incorrecto;
- `ESQUEMA_TEMPERATURAS`, con Pandera;
- `validar_datos` y `casos_que_fallan`.

In [ ]:
from src.meteolab.esquema import (
    ESQUEMA_TEMPERATURAS,
    casos_que_fallan,
    validar_datos,
    validar_esquema,
)

validar_esquema(lecturas_modulo)
validado = validar_datos(lecturas_modulo)
print("Filas validadas:", validado.height)
print(ESQUEMA_TEMPERATURAS)

In [ ]:

fallas = casos_que_fallan(lecturas_modulo)
if fallas.height > 0 and "index" in fallas.columns:
    indices_invalidos = fallas.get_column("index").drop_nulls().unique().to_list()
else:
    indices_invalidos = []
muestra_invalida = (
    lecturas_modulo
    .with_row_index("indices")
    .filter(pl.col("indices").is_in(indices_invalidos))
    .drop("indices")
)

---
# Etapa 4 — Limpieza: conservar solo meses (0.9 puntos)

Esta es la decisión central del laboratorio. El archivo mezcla tres escalas
temporales. Desde este punto trabajarán solo con las temperaturas medias de
`JAN` a `DEC`.

In [ ]:
from src.meteolab.constantes import (
    PERIODOS_MENSUALES,
)

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Contrato de limpieza</strong>
  La tabla limpia debe contener únicamente los 12 períodos mensuales. Debe descartar <code>PERIODOS_ESTACIONALES = ("DJF", "MAM", "JJA", "SON")</code> y <code>PERIODO_ANUAL = "ANN"</code>. En este dataset, el nulo conocido pertenece a <code>DJF</code> de 2025 y desaparece al aplicar la selección mensual.
</div>

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — filtrar filas</strong>
  Filtrar una tabla significa conservar las filas que cumplen una condición. Una condición sobre <code>period</code> define el universo temporal del análisis; no es lo mismo que corregir un valor ni que eliminar una columna.
</div>


### 4.1 — Investigar y limpiar (0.3 puntos)

Investiguen cómo combinar condiciones con `&` y cómo comprobar nulos. Luego,
filtren en el notebook la tabla para conservar solo `PERIODOS_MENSUALES` y
valores disponibles de `temperature_c`. Comprueben las filas y el esquema
resultantes.


In [ ]:
limpias = lecturas_modulo.filter((pl.col("period").is_in(PERIODOS_MENSUALES)) & (pl.col("temperature_c").is_not_null()))

In [ ]:
display(limpias.null_count())

In [ ]:
assert set(limpias["period"].unique()) <= {
    "JAN",
    "FEB",
    "MAR",
    "APR",
    "MAY",
    "JUN",
    "JUL",
    "AUG",
    "SEP",
    "OCT",
    "NOV",
    "DEC",
}
assert limpias["temperature_c"].null_count() == 0
print("Filas mensuales limpias:", limpias.height)

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 6 — No mezclar escalas (0.1 puntos)</strong>
  ¿Qué problema produciría calcular una media por país usando a la vez meses, estaciones y `ANN`? Expliquen qué filas quedan en la tabla limpia y qué representa cada una.
</div>

<code>Escribe tu respuesta aquí:</code> En la tabla limpia quedan todas las filas que no tienen valores null y sólo usando los meses en vez de estaciones o anuales. Cada una representa la observación mensual más "pequeña" en consideraciones de tiempo cuya temperatura tenga un valor que no se haya perdido. Ahora, en principio no habría problema si las estaciones calculan el promedio desde los promedios mensuales y el promedio anual también es un promedio de los resultados mensuales,  porque en este caso no se agregaría nueva información sino que sería una suma ponderada de los promedios mensuales ($\frac{12 \overline{meses}}{17}+\frac{4 \overline{estaciones}}{17}+\frac{ANN}{17}$). El problema podría ocurrir cuando los promedios por estaciones o anual no se hacen de manera mensual pues en ese caso la fórmula anterior podría no responder de buena manera porque la manera que pondera la información podría arruinar todo. Además, podría dar el caso de valores nulos en los meses y eso podría podría echar abajo los cálculos de promedios anuales (digamos, si falta toda la información del invierno). 

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 7 — Comprobar la completitud mensual (0.1 puntos)</strong>
  Después de la limpieza, ¿cómo comprobarían que cada combinación de país y año tiene doce observaciones mensuales? ¿Qué decisión tomarían si faltara un mes?
</div>

<code>Escribe tu respuesta aquí:</code> La idea sería agrupar el dataset limpio por país y año, luego la idea sería contar cuántos NO están en los posibles meses. Algo así

<code>
meses_esperados = {
    "JAN", "FEB", "MAR", "APR", "MAY", "JUN",
    "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"
}

faltantes = (
    limpias
    .group_by(["country", "year"])
    .agg(
        pl.col("period").unique().alias("meses")
    )
    .with_columns(
        pl.col("meses")
        .list.set_difference(pl.lit(list(meses_esperados)))
        .alias("meses_faltantes")
    )
    .filter(pl.col("meses_faltantes").list.len() > 0)
)
</code>

En el caso de que faltantes no sea vacío y sea necesario tomar una decisión por los casos faltantes, revisaría primero cuántos o cuáles meses faltan. Si faltan pocos meses o de manera "aislada", me haría mucho sentido poder hacer imputación. Una dificultad sin embargo, es cómo hacerlo; si "aislado" significa que en años anteriores o siguientes no ha ocurrido, tomaría un promedio del año anterior y siguiente. La motivación es preservar un sentido de estacionalidad. En caso de que no estuviera disponible alguno de los valores del año siguiente o anterior o que consistentemente no esté disponible ese mes o grupo de meses y suponiendo que no se puede imputar por $ANN$ o el valor de estacionalidad por no disponibilidad (y porque arruina todo el ejercicio, si queremos saber el promedio anual, imputar por el mismo está en contra de la metodología), entonces lo mejor sería no agregar al análisis esos años difíciles porque cualquier imputación distorcionaría mucho el valor real del año y sería poco útil para un análisis posterior.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 8 — Imputar con el promedio (0.2 puntos)</strong>
  El nulo conocido pertenece a `DJF` de 2025. Calculen el promedio global de `temperature_c` y supongan que, antes de filtrar los períodos, reemplazan ese valor por dicho promedio. ¿Qué valor se asignaría, qué pasaría con la tabla y qué problema introduciría esa imputación? Expliquen por qué en este laboratorio es preferible descartar la fila.
</div>

<code>Escribe tu respuesta aquí:</code> El problema es que esta imputación no respeta la estructura de los datos. El promedio global fue calculado mezclando períodos mensuales, estacionales y anuales, por lo que no necesariamente representa una temperatura apropiada para la estación DJF. Además, reemplazar hace parecer que se tiene una observación real cuando en realidad el valor fue estimado.

Luego se conservarán únicamente los períodos mensuales (JAN, FEB, ..., DEC). Por lo tanto, la observación DJF será descartada de todas formas. Imputarla previamente no entrega información útil para el análisis y modifica innecesariamente los datos originales.

Por estas razones, en este caso es preferible simplemente descartar la fila con el valor nulo, especialmente considerando que pertenece a un período estacional que no será utilizado posteriormente.

### 4.2 — Trasladar la limpieza al módulo (0.2 puntos)

Después de probar las expresiones en el notebook, implementen en
`src/meteolab/limpieza.py`:

- `limpiar_temperaturas`, compatible con `DataFrame` y `LazyFrame`;
- `resumen_de_nulos`, para revisar los valores faltantes;
- `claves_repetidas`, para detectar repeticiones de país, año y período.

Las tres funciones se utilizan en las pruebas de esta etapa.

In [ ]:
from src.meteolab.limpieza import limpiar_temperaturas, resumen_de_nulos

limpias_modulo = limpiar_temperaturas(lecturas_modulo)

---
# Etapa 5 — Fechas y agregaciones mensuales (1.1 puntos)

`year` y `period` son columnas separadas. Construirán una fecha para ordenar
las observaciones y luego resumirán los años disponibles por mes.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — columna derivada</strong>
  Una columna derivada se calcula a partir de columnas existentes. La columna <code>fecha</code> no agrega una medición: combina <code>year</code> y el número del mes para permitir ordenamiento y gráficos temporales.
</div>

### 5.1 — Investigar y construir fechas (0.2 puntos)

Investiguen cómo traducir los códigos `JAN` a `DEC` a números de mes y cómo
convertir una cadena con formato ISO en una fecha de Polars. Construyan en el
notebook las columnas `month` y `fecha`, y comprueben que el resultado se
ordena cronológicamente.

In [ ]:
from src.meteolab.constantes import MESES

In [ ]:
mensuales = limpias_modulo.with_columns(
    pl.col("period")
      .replace(MESES)
      .cast(pl.Int8)
      .alias("month")
).with_columns(
    (
        pl.col("year").cast(pl.String)
        + "-"
        + pl.col("month").cast(pl.String).str.zfill(2)
        + "-01"
    )
    .str.to_date("%Y-%m-%d")
    .alias("fecha")
)

### 5.2 — Investigar y calcular resúmenes (0.4 puntos)

El resultado de esta exploración se trasladará después a
`src/meteolab/metricas.py`, en `resumen_mensual`. Debe devolver una fila por
país y mes, con:

- `iso_alpha3` y `country`;
- `month`;
- `observaciones`;
- `temperature_mean`, redondeada a dos decimales.

Antes de escribir la función, investiguen `group_by` y `agg`. Calculen el
resumen directamente en el notebook y revisen algunas filas.

In [ ]:
climatologia = (
    mensuales
    .group_by(["iso_alpha3", "country", "month"])
    .agg(
        pl.len().alias("observaciones"),
        pl.col("temperature_c").mean().round(2).alias("temperature_mean"),
    )
    .sort(["iso_alpha3", "month"])
)

In [ ]:
PAISES = ["CHL", "ARG", "PER", "BOL", "BRA", "CAN", "EGY"]
fig = px.line(
    climatologia.filter(pl.col("iso_alpha3").is_in(PAISES)),
    x="month",
    y="temperature_mean",
    color="country",
    markers=True,
    title="Climatología mensual de países seleccionados",
    labels={
        "month": "Mes",
        "temperature_mean": "Temperatura media (°C)",
        "country": "País",
    },
)
fig.show()

Calculen también una media por año, pero debe salir de las filas mensuales
limpias. No usen las filas `ANN` del CSV. El resumen debe conservar los años
incompletos y la columna `meses_disponibles`; para comparar períodos, filtren
después los años cuyo conteo sea igual a 12.

In [ ]:
anuales_desde_meses = (
    mensuales
    .group_by(["iso_alpha3", "country", "year"])
    .agg(
        pl.col("temperature_c").mean().round(2).alias("temperature_mean"),
        pl.col("month").n_unique().alias("meses_disponibles"),
    )
    .sort(["iso_alpha3", "year"])
)

In [ ]:
anuales_completos = anuales_desde_meses.filter(
    pl.col("meses_disponibles") == 12
)
print("Años completos:", anuales_completos.height)

Revisen si la tabla contiene más de una observación para la misma combinación
de país, año y período. Investiguen `group_by` y `len` para contar posibles
repeticiones de esa clave.

In [ ]:
repetidas = (
    mensuales
    .group_by(["iso_alpha3", "year", "period"])
    .len()
    .filter(pl.col("len") > 1)
)

In [ ]:
repetidas

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 9 — Qué significa una climatología mensual (0.2 puntos)</strong>
  En `climatologia`, ¿qué representa una fila para `CHL` y `month = 1`? ¿Por qué esa fila resume varios años y no un solo registro del archivo?
</div>

<code>Escribe tu respuesta aquí:</code> Es el promedio para chile en el mes de enero desde 1901 a 2025. Es el resumen porque se obtuvo de promediar en grupo la temperatura. 

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 10 — Detectar duplicados (0.2 puntos)</strong>
  ¿Qué combinación de columnas debería identificar de forma única una observación mensual? ¿Cómo detectarían registros duplicados antes de calcular las métricas?
</div>

<code>Escribe tu respuesta aquí:</code> La identificación única debería hacerse usando (iso_alpha3, year, month). Los duplicados puede ser como se implementó antes, pero también con el método claves_repetidas  

---

### 5.3 — Trasladar las transformaciones al módulo (0.1 puntos)

Después de probar las expresiones, implementen `agregar_fecha_mensual`,
`resumen_mensual` y `resumen_anual_desde_mensuales` en sus módulos. Comparen
los resultados del módulo con los que obtuvieron directamente en el
notebook.

In [ ]:
from src.meteolab.derivadas import agregar_fecha_mensual
from src.meteolab.metricas import (
    resumen_anual_desde_mensuales,
    resumen_mensual,
)
from src.meteolab.limpieza import claves_repetidas

In [ ]:
mensuales_modulo = agregar_fecha_mensual(limpias_modulo)
climatologia_modulo = resumen_mensual(mensuales_modulo)
anuales_modulo = resumen_anual_desde_mensuales(mensuales_modulo, ["CHL"])
repetidas_modulo = claves_repetidas(mensuales_modulo)

---
# Etapa 6 — Ventanas y anomalías mensuales (0.8 puntos)

Una media histórica de enero no debe compararse con una de julio. Usarán una
ventana por país y mes para medir cuánto se aparta cada observación de sus
pares comparables.

<!-- DEFINICIÓN -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #3fb950; background:rgba(63, 185, 80,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📖 Definición — ventana</strong>
  Una expresión con <code>.over("iso_alpha3", "month")</code> calcula una medida dentro de cada grupo y devuelve ese resultado en las filas originales. A diferencia de <code>group_by().agg()</code>, una ventana conserva una fila por observación.
</div>

### 6.1 — Investigar y calcular una ventana (0.4 puntos)

Investiguen `over` y construyan directamente en el notebook las tres
columnas solicitadas. Usen la dupla `(iso_alpha3, month)` como grupo de
comparación para que cada mes se compare con otros del mismo país.

Implementen `anomalias_mensuales(mensuales, umbral=2.0)` en el módulo después
de comprobar el resultado de esta exploración. Debe agregar:

- `temperature_mean_month`, la media histórica del país para ese mes;
- `standardized_anomaly`, la diferencia dividida por la desviación estándar;
- `is_anomaly`, booleana y sin nulos.

In [ ]:
marcadas = mensuales.with_columns(
    pl.col("temperature_c")
      .mean()
      .over(["iso_alpha3", "month"])
      .alias("temperature_mean_month")
).with_columns(
    (
        (pl.col("temperature_c") - pl.col("temperature_mean_month"))
        /
        pl.col("temperature_c")
          .std()
          .over(["iso_alpha3", "month"])
    )
    .alias("standardized_anomaly")
).with_columns(
    (
        pl.col("standardized_anomaly").abs() > 2.0
    )
    .fill_null(False)
    .alias("is_anomaly")
)

In [ ]:
fig = px.scatter(
    marcadas.filter(pl.col("iso_alpha3").is_in(PAISES)),
    x="fecha",
    y="standardized_anomaly",
    color="is_anomaly",
    facet_row="iso_alpha3",
    hover_data=["country", "period", "temperature_c"],
    title="Anomalías mensuales",
    labels={
        "fecha": "Fecha",
        "standardized_anomaly": "Anomalía estandarizada",
        "is_anomaly": "¿Anómala?",
    },
)
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 11 — Elegir el grupo de comparación (0.3 puntos)</strong>
  ¿Qué diferencia habría entre calcular la anomalía sobre `iso_alpha3` y calcularla sobre `(iso_alpha3, month)`? ¿Cuál de las dos opciones permite distinguir mejor una variación inusual de una diferencia normal entre estaciones del año?
</div>

<code>Escribe tu respuesta aquí:</code> En el caso de hacer algo como .mean().over("iso_alpha3") la media de Chile mezcla: 22,21,23,8,7,9,... . Por lo tanto, una temperatura de 22 °C en enero podría aparecer como una anomalía positiva simplemente porque Chile normalmente tiene temperaturas menores durante el invierno. .mean().over(["iso_alpha3", "month"]) hace que los 22 °C de enero se comparen solamente con enero

---

### 6.2 — Trasladar la ventana al módulo (0.1 puntos)

Después de revisar el resultado, implementen `anomalias_mensuales` en
`src/meteolab/metricas.py`. Comparen las columnas y la cantidad de filas con
el cálculo realizado directamente en el notebook.

In [ ]:
from src.meteolab.metricas import anomalias_mensuales

marcadas_modulo = anomalias_mensuales(mensuales_modulo, umbral=2.0)

---
# Etapa 7 — Pipeline lazy y análisis (1.1 puntos)

Integrarán las funciones en una consulta lazy. El pipeline debe leer el CSV,
descartar los períodos no mensuales, construir las fechas y producir el
resumen pedido sin materializar pasos intermedios.

<!-- IMPORTANTE -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #8957e5; background:rgba(137, 87, 229,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">📌 Contrato del pipeline</strong>
  <code>pipeline_mensual</code> debe devolver un <code>LazyFrame</code>. El filtrado de países, la selección de meses y la construcción de columnas deben formar parte del plan. La ejecución ocurre solo con <code>collect()</code>.
</div>

### 7.1 — Investigar y construir una consulta lazy (0.1 puntos)

Investiguen cómo combinar `scan_csv`, `filter`, `with_columns` y `collect`.
Construyan primero una consulta lazy directamente en el notebook que:

- lea el CSV con los tipos esperados;
- conserve solo las filas mensuales con temperatura disponible;
- seleccione los países de `PAISES`;
- agregue `month` y `fecha`;
- ordene por país y fecha.

Comprueben que la consulta no se ejecuta hasta llamar a `collect()` y revisen
las primeras filas del resultado.

In [ ]:
consulta_notebook = (
    pl.scan_csv(
        RUTA_CSV,
        schema_overrides={
            "country": pl.String,
            "iso_alpha2": pl.String,
            "iso_alpha3": pl.String,
            "year": pl.Int64,
            "period": pl.String,
            "temperature_c": pl.Float64,
            "parameter": pl.String,
            "units": pl.String,
            "source_file": pl.String,
        },
    )
    .filter(
        pl.col("period").is_in(PERIODOS_MENSUALES)
        & pl.col("temperature_c").is_not_null()
        & pl.col("iso_alpha3").is_in(PAISES)
    )
    .with_columns(
        pl.col("period")
        .replace(MESES)
        .cast(pl.Int8)
        .alias("month")
    )
    .with_columns(
        pl.date(
            pl.col("year"),
            pl.col("month"),
            1,
        ).alias("fecha")
    )
    .sort(["iso_alpha3", "fecha"])
)

In [ ]:
consulta_notebook

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 12 — Elegir entre lazy y eager (0.2 puntos)</strong>
  Comparen esta consulta lazy con una solución eager que lea el archivo mediante `read_csv` y ejecute cada transformación de inmediato. ¿Qué ventajas ofrece construir un `LazyFrame` y ejecutar al final con `collect()`? Mencionen dos ventajas concretas y una situación en que preferirían trabajar en modo eager.
</div>

<code>Escribe tu respuesta aquí:</code> Una ventaja es que Polars puede optimizar el plan completo antes de ejecutarlo, por ejemplo reordenando filtros y evitando operaciones innecesarias. Otra ventaja es que se puede trabajar con archivos grandes sin materializar cada resultado intermedio en memoria, dejando la ejecución para el momento en que se llama a collect(). En cambio, preferiría el modo eager cuando estoy trabajando con un conjunto de datos pequeño, haciendo una exploración interactiva o cuando necesito inspeccionar inmediatamente el resultado de cada transformación.

### 7.2 — Trasladar el pipeline al módulo (0.2 puntos)

Después de probar la consulta, implementen en `src/meteolab/reporte.py`:

- `pipeline_mensual`;
- `pipeline_resumen_mensual`;
- `pipeline_resumen_anual`, calculado desde meses;
- `pipeline_anomalias`;
- `ejecutar_reporte` y `plan_de_ejecucion`.

In [ ]:
from src.meteolab.reporte import (
    ejecutar_reporte,
    pipeline_anomalias,
    pipeline_mensual,
    pipeline_resumen_anual,
    pipeline_resumen_mensual,
    plan_de_ejecucion,
)

consulta = pipeline_mensual(RUTA_CSV, PAISES)
print(type(consulta))
print(plan_de_ejecucion(RUTA_CSV, PAISES))
print("Filas directas:", consulta_notebook.collect().height)
print("Filas del módulo:", consulta.collect().height)

In [ ]:
reporte_mensual = ejecutar_reporte(RUTA_CSV, PAISES)
print(reporte_mensual.shape)
display(reporte_mensual.head())

In [ ]:
reporte_anual = pipeline_resumen_anual(RUTA_CSV, PAISES).collect()
anomalias = pipeline_anomalias(RUTA_CSV, ["CHL"]).collect()
print("Años calculados desde meses:", reporte_anual.height)
print("Anomalías marcadas:", anomalias["is_anomaly"].sum())

### 7.3 — Analizar cambios de temperatura a largo plazo (0.2 puntos)

Construyan primero una serie con la media anual de cada país y grafiquen cómo
evoluciona desde 1901. Después comparen la media de 1901–1930 con la de
1991–2020. Estas comparaciones describen cambios de temperatura; por sí solas
no demuestran sus causas.

In [ ]:
from src.meteolab.constantes import PAISES_COMPARACION

evolucion_anual = (
    resumen_anual_desde_mensuales(
            agregar_fecha_mensual(limpias_modulo),
            PAISES_COMPARACION
        )
    .sort(["iso_alpha3", "year"])
    .filter(pl.col("meses_disponibles") == 12)
    .sort(["country", "year"])
)

In [ ]:
fig = px.line(
    evolucion_anual,
    x="year",
    y="temperature_mean",
    color="country",
    title="Evolución de la temperatura media anual",
    labels={
        "year": "Año",
        "temperature_mean": "Temperatura media (°C)",
        "country": "País",
    },
)
fig.show()

Comparen ahora los dos períodos de referencia para resumir el cambio.

In [ ]:
# Su código aquí: calculen `comparacion` y agreguen `cambio_c`.
comparacion = (
    evolucion_anual
    .with_columns(
        pl.when(pl.col("year").is_between(1901, 1930))
        .then(pl.lit("1901-1930"))
        .when(pl.col("year").is_between(1991, 2020))
        .then(pl.lit("1991-2020"))
        .otherwise(None)
        .alias("periodo")
    )
    .filter(pl.col("periodo").is_not_null())
    .group_by(["iso_alpha3", "country", "periodo"])
    .agg(
        pl.col("temperature_mean")
        .mean()
        .round(2)
        .alias("temperature_mean")
    )
    .pivot(
        on="periodo",
        index=["iso_alpha3", "country"],
        values="temperature_mean",
    )
    .with_columns(
        (
            pl.col("1991-2020") - pl.col("1901-1930")
        )
        .round(2)
        .alias("cambio_c")
    )
    .sort("cambio_c", descending=True)
)

In [ ]:
comparacion

In [ ]:
fig = px.bar(
    comparacion,
    x="country",
    y="cambio_c",
    color="cambio_c",
    title="Cambio de temperatura media entre períodos de referencia",
    labels={
        "country": "País",
        "cambio_c": "Cambio de temperatura (°C)",
    },
)
fig.update_xaxes(categoryorder="total descending")
fig.show()

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 13 — Cambios de temperatura y sus límites (0.2 puntos)</strong>
  Observen la evolución anual y la tabla <code>comparacion</code>. ¿Qué países presentan el mayor aumento entre los dos períodos? ¿Todos muestran el mismo cambio? Usen valores concretos para describir la tendencia y expliquen por qué este análisis aporta evidencia descriptiva para estudiar el calentamiento global, pero no permite atribuir causas ni calcular por sí solo una temperatura global.
</div>

<code>Escribe tu respuesta aquí:</code> Los países presentan aumentos de temperatura media entre los períodos 1901–1930 y 1991–2020, aunque la magnitud del cambio no es igual para todos. El mayor aumento se observa en Canadá, con 1,28 °C, seguido por Egipto, con 0,92 °C, y Brasil, con 0,83 °C. Argentina presenta un aumento de 0,47 °C, mientras que Perú y Chile muestran aumentos de 0,36 °C y 0,29 °C, respectivamente. Bolivia presenta el menor cambio, con 0,08 °C. Por lo tanto, aunque todos los países analizados muestran un aumento entre ambos períodos, existen diferencias importantes en su magnitud.

Esta comparación aporta evidencia descriptiva de un cambio de temperatura a largo plazo, ya que permite observar que las temperaturas medias de los períodos más recientes son superiores a las del período inicial en los países analizados. Sin embargo, estos resultados por sí solos no permiten determinar las causas de estos cambios ni atribuirlos directamente a un factor específico. Además, tampoco permiten calcular una temperatura global, ya que el análisis considera solamente los países incluidos en el archivo y no constituye una estimación global de la temperatura del planeta.

<!-- PREGUNTA -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #58a6ff; background:rgba(88, 166, 255,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">❓ Pregunta 14 — Interpretar el resultado (0.2 puntos)</strong>
  Elijan un país y describan dos patrones que observen en su climatología mensual o en sus anomalías. Usen valores concretos de las tablas o gráficos. Indiquen también qué información no se puede concluir a partir de este archivo.
</div>

In [ ]:
climatologia_chile = (
    climatologia_modulo
    .filter(pl.col("iso_alpha3") == "CHL")
    .sort("month")
)

climatologia_chile.sort(
    "temperature_mean",
    descending=True
)

In [ ]:
climatologia_chile.select(
    [
        pl.col("temperature_mean").max().alias("maximo"),
        pl.col("temperature_mean").min().alias("minimo"),
        (
            pl.col("temperature_mean").max()
            - pl.col("temperature_mean").min()
        ).round(2).alias("amplitud")
    ]
)

<code>Escribe tu respuesta aquí:</code> La climatología mensual de Chile presenta una variación estacional clara. El mes más cálido presenta una temperatura media de 13.08°C, mientras que el más frío alcanza 4.91°C, dando una diferencia de 8.17°C entre ambos. Esto muestra que la temperatura media de Chile no se mantiene constante durante el año, sino que presenta un ciclo asociado a los meses. (sigue)

In [ ]:
anomalias_chile = (
    marcadas_modulo
    .filter(pl.col("iso_alpha3") == "CHL")
)

In [ ]:
anomalias_chile.sort(
    "standardized_anomaly",
    descending=True
).select(
    [
        "year",
        "period",
        "temperature_c",
        "temperature_mean_month",
        "standardized_anomaly",
        "is_anomaly",
    ]
).head(5)

In [ ]:
anomalias_chile.sort(
    "standardized_anomaly"
).select(
    [
        "year",
        "period",
        "temperature_c",
        "temperature_mean_month",
        "standardized_anomaly",
        "is_anomaly",
    ]
).head(5)

In [ ]:
anomalias_chile.select(
    pl.col("is_anomaly").sum().alias("cantidad_anomalias")
)

Además de la estacionalidad, se observan meses cuya temperatura se aleja considerablemente de la media histórica correspondiente a ese mes. Por ejemplo, en 2022 noviembre, la temperatura fue de 12.4°C, en contraste a la media histórica de 10.51 °C, produciendo una anomalía estandarizada de Z. Este valor supera el umbral de 2 utilizado en el análisis, por lo que fue clasificado como anómalo.

A partir de este archivo no se puede concluir cuál es la causa de las variaciones observadas ni atribuirlas directamente a una causa particular. Tampoco se puede determinar por sí solo que un evento anómalo sea consecuencia del cambio climático o del fenómeno del niño o de la niña. Además, el archivo contiene temperaturas medias correspondientes a determinados países y no una medición global espacialmente completa, por lo que no permite calcular directamente una temperatura media global.

<!-- MINI PROYECTO -->
<div style="padding:16px 20px; margin:16px 0; border-left:4px solid #f0883e; background:rgba(240, 136, 62,.12); color:inherit; border-radius:4px;">

  <strong style="display:block; margin-bottom:8px;">🛠️ Antes de entregar</strong>
  Ejecuten las pruebas por etapa y luego la suite completa:

  <pre><code>uv run pytest -m etapa1
uv run pytest -m etapa2
uv run pytest -m etapa3
uv run pytest -m etapa4
uv run pytest -m etapa5
uv run pytest -m etapa6
uv run pytest -m etapa7
uv run pytest</code></pre>

  Reinicien el kernel y ejecuten el notebook completo. Revisen que los gráficos Plotly se rendericen y que las respuestas usen resultados observables.
</div>